# Last.fm Global Trends — Explorer

Quick exploration of the `trends.db` dataset using DuckDB and pandas.

- **Global** — top artists, tracks, and tags worldwide
- **By Country** — top artists and tracks for any supported country

In [ ]:
!pip install -q duckdb

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DB_PATH = "/kaggle/input/last-fm-global-trends/trends.db"

con = duckdb.connect(DB_PATH, read_only=True)

# Quick sanity check
for table in ["global_top_artists", "global_top_tracks", "global_top_tags", "geo_top_artists", "geo_top_tracks"]:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count:,} rows")

## Global Top Artists

In [ ]:
top_artists = con.execute("""
    SELECT rank AS "Rank", artist AS "Artist",
           listeners AS "Listeners", playcount AS "Scrobbles"
    FROM global_top_artists
    ORDER BY rank
    LIMIT 25
""").df()

top_artists[["Listeners", "Scrobbles"]] = top_artists[["Listeners", "Scrobbles"]].apply(
    lambda c: c.map("{:,}".format)
)
top_artists.set_index("Rank")

### Top 20 Artists by Scrobbles

In [ ]:
chart_df = con.execute("""
    SELECT artist, playcount
    FROM global_top_artists
    ORDER BY playcount DESC
    LIMIT 20
""").df().sort_values("playcount")

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(chart_df["artist"], chart_df["playcount"], color="#16a34a")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e9:.1f}B" if x >= 1e9 else f"{x/1e6:.0f}M"))
ax.set_xlabel("Scrobbles")
ax.set_title("Top 20 Global Artists by Scrobbles")
plt.tight_layout()
plt.show()

## Global Top Tracks

In [ ]:
top_tracks = con.execute("""
    SELECT rank AS "Rank", track AS "Track", artist AS "Artist",
           listeners AS "Listeners", playcount AS "Scrobbles"
    FROM global_top_tracks
    ORDER BY rank
    LIMIT 25
""").df()

top_tracks[["Listeners", "Scrobbles"]] = top_tracks[["Listeners", "Scrobbles"]].apply(
    lambda c: c.map("{:,}".format)
)
top_tracks.set_index("Rank")

### Artists With Most Tracks in Global Top 10,000

In [ ]:
artist_counts = con.execute("""
    SELECT artist AS "Artist", COUNT(*) AS "Tracks in Top 10k"
    FROM global_top_tracks
    GROUP BY artist
    ORDER BY 2 DESC
    LIMIT 20
""").df()

artist_counts.index = range(1, len(artist_counts) + 1)
artist_counts.index.name = "Rank"
artist_counts

## Global Top Tags

In [ ]:
top_tags = con.execute("""
    SELECT rank AS "Rank", tag AS "Tag",
           reach AS "Reach", taggings AS "Taggings"
    FROM global_top_tags
    ORDER BY rank
    LIMIT 25
""").df()

top_tags[["Reach", "Taggings"]] = top_tags[["Reach", "Taggings"]].apply(
    lambda c: c.map("{:,}".format)
)
top_tags.set_index("Rank")

## By Country

Change `COUNTRY` to any supported country name (e.g. `"Brazil"`, `"Japan"`, `"Germany"`).

In [ ]:
COUNTRY = "United States"

# Available countries
available = con.execute(
    "SELECT DISTINCT country FROM geo_top_artists ORDER BY country"
).df()["country"].tolist()
print(f"{len(available)} countries available")
print(", ".join(available[:20]), "...")

### Top Artists by Country

In [ ]:
geo_artists = con.execute("""
    SELECT rank AS "Rank", artist AS "Artist", listeners AS "Listeners"
    FROM geo_top_artists
    WHERE country = ?
    ORDER BY rank
    LIMIT 25
""", [COUNTRY]).df()

geo_artists["Listeners"] = geo_artists["Listeners"].map("{:,}".format)
geo_artists.set_index("Rank")

### Top Tracks by Country

In [ ]:
geo_tracks = con.execute("""
    SELECT rank AS "Rank", track AS "Track", artist AS "Artist", listeners AS "Listeners"
    FROM geo_top_tracks
    WHERE country = ?
    ORDER BY rank
    LIMIT 25
""", [COUNTRY]).df()

geo_tracks["Listeners"] = geo_tracks["Listeners"].map("{:,}".format)
geo_tracks.set_index("Rank")

### Top 20 Artists by Listeners — Country Chart

In [ ]:
geo_chart = con.execute("""
    SELECT artist, listeners
    FROM geo_top_artists
    WHERE country = ?
    ORDER BY listeners DESC
    LIMIT 20
""", [COUNTRY]).df().sort_values("listeners")

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(geo_chart["artist"], geo_chart["listeners"], color="#1b6ef3")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K"))
ax.set_xlabel("Listeners")
ax.set_title(f"Top 20 Artists in {COUNTRY} by Listeners")
plt.tight_layout()
plt.show()

In [ ]:
con.close()